# A tour of ercot-mis

> **This notebook pulls ERCOT Secure and ECEII data into `demo/data/` (gitignored) and shows it on your
> screen.** Equipment, bus and substation names are confidential: do not paste or screenshot the outputs,
> and strip them before you commit this file. The pre-commit guard refuses a notebook with outputs:
>
>     uv run --with nbstripout nbstripout demo/tour.ipynb

Run it with `uv run --with jupyterlab jupyter lab demo/tour.ipynb` from the repository root, after
`uv sync`, with `.env` and the certificate in place (CLAUDE.md, "Environment"). Every code cell is a
call you would make yourself. The text before each cell says what the call does.

**What ercot-mis is.** A fetcher. It downloads ERCOT's network models and the inputs around them,
keeps the original bytes, parses them into typed tables and standardizes them, so that every table
traces back to the bytes ERCOT published. It tests its models against what the market cleared.
Analysis lives elsewhere.

**Words used below.**

- **node**: a point of a node-breaker model, which is one record of the CRR RAW file: the finest point its closed breakers separate.
- **bus**: a point of a bus-branch model, which is one record of the DAM RAW file, or the CRR nodes that closed breakers join into one electrical point. `bus_key` identifies a bus. PSS/E calls every record a bus; only the raw layer keeps that word.
- **substation**: the physical yard, holding buses at several voltage levels.
- **settlement point**: a place ERCOT prices and settles at: a resource node (one bus), a hub (an average over its hub buses) or a load zone (a load-weighted average over its loads' buses).
- **contingency**: an outage the market secures against: a line, a transformer, or a bus split in two.
- **shift factor**: the share of one MW injected at a bus that flows on a given branch. A row of shift factors is one branch's sensitivity to every bus. ERCOT's prices are built from them.
- **shadow price**: what one more MW of room on a binding constraint would have been worth.
- **GTC**, generic transmission constraint: a limit ERCOT places on the flows of several branches together. **GTL** is its daily limit.
- **EMIL id** (`NP4-500-SG` and the like): ERCOT's catalogue number for a data product.
- **PSS/E** and **RAW file**: a power-flow program and its text case format; both ERCOT models are PSS/E RAW files.
- **DC power flow**: the linear model of how power splits over branches by their reactances; ERCOT's market engines price with it, and so does this repository.
- **slack**: the bus a DC solve measures everything against. It does not change flows; it is the bus with a sensitivity of zero.
- **tie**: in the CRR model, a closed breaker or switch written as a line at PSS/E's minimum reactance. The auction solves ties as branches.
- **electrical bus**: ERCOT's name for a pricing point in its price and mapping files; only the raw and mapping tables use the term.
- **time-of-use block** (`PeakWD`, `PeakWE`, `Off-peak`): the CRR auction's hour groups; CRR ratings are given per block.
- **DC tie**: a converter station linking ERCOT to a neighbouring grid; a settlement point of its own kind.
- **snapshot**: one model, with a revision number: `crr:monthly:2026-10:r1`, `dam:2026-10-05:he12:r1`.


In [ ]:
from datetime import date, timedelta
from pathlib import Path

import polars as pl

import ercot_mis as em

pl.Config.set_tbl_rows(30)
REPO = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA = REPO / "demo" / "data"  # a data folder of its own, so the tour never touches data/
s = em.open(DATA)
s.data_dir

## 1. Products

`em.PRODUCTS` lists the ERCOT data products the repository knows. Two sources deliver them.
**EWS** (External Web Services) is ERCOT's certificate-authenticated service for market participants.
It carries the confidential network models, and it keeps a document only for the product's
**display window**: what is not archived in time is gone. **The Public API** is ERCOT's public
archive; it keeps years. A product is **fetched** (downloaded) or only **listed**, so that what ERCOT
offers is on record. `classification` says who may see it: Public, Secure, or ECEII.


In [ ]:
pl.DataFrame([{"emil_id": p.emil_id, "name": p.name, "classification": p.classification, "source": p.source,
               "is_fetched": p.is_fetched, "window_days": p.display_days} for p in em.PRODUCTS.values()])

## 2. Listing and the archive

`list` asks ERCOT what it offers for a product now and records every document in the local
**catalog**, a DuckDB file. It never returns ERCOT's file names, which embed the participant's DUNS.
`fetch` downloads what is listed and not yet archived.

The **archive** keeps the original zips, named by the SHA-256 of their bytes, read-only and never
unzipped to disk. A re-posted document is a new document, not an overwrite. The catalog records every
listing, every blob, every zip member and every table built from them: the chain from a row back to
ERCOT's bytes.


In [ ]:
docs = s.list("NP7-800-M")  # the monthly CRR network model
docs.select("doc_id", "posted_at", "size_bytes", "is_archived").tail(5)

The fetch below takes one DAM day (24 hourly models), the newest monthly CRR model, the GTC limits
workbook, ERCOT's settlement point mappings, the day's prices and pricing inputs, and one 60-day
disclosure document. `operating_dates` keeps only the listing days you name; the DAM package posted on
a day holds the next day's models. `since` bounds the posting time. Each call returns one row per
document with its status, and a failed download is retried on the next call.


In [ ]:
DAY = date.today() - timedelta(days=1)  # the DAM package posted yesterday holds today's 24 hourly models
pulls = {
    "NP4-500-SG": dict(operating_dates=[DAY]),
    "NP7-800-M": dict(since=DAY - timedelta(days=45)),
    "NP3-766-M": dict(since=DAY - timedelta(days=2)),
    "NP3-220-SG": dict(since=DAY - timedelta(days=20)),
    "NP4-160-SG": dict(since=DAY - timedelta(days=20)),
    "NP3-966-ER": dict(since=DAY - timedelta(days=1)),
}
for emil_id in ("NP4-191-CD", "NP4-183-CD", "NP4-190-CD", "NP4-523-CD", "NP4-200-CD", "NP4-158-SG", "NP4-231-CD"):
    pulls[emil_id] = dict(since=DAY - timedelta(days=1))
for emil_id, kwargs in pulls.items():
    result = s.fetch(emil_id, **kwargs)
    print(f"{emil_id:11s}", dict(result.group_by("status").len().rows()), f"{result['size_bytes'].sum() / 1e6:.1f} MB")

## 3. The raw layer

`build_raw` parses every archived package of a product into Parquet tables under `raw/`, one table per
kind of source file, named for the file (`psse_bus`, `dam_lines`, `crr_contingencies`). Column names
become snake_case and values are typed; nothing else changes. Every row carries identity columns
(`emil_id`, `doc_id`, `blob_sha256`, `member_path`, and the package's own keys such as `operating_date`
and `hour`). The catalog records each written file keyed by the parser version and the blob, so a rerun
skips what is built and a parser change rebuilds exactly what that parser produced.

The PSS/E RAW files are read by the repository's own parser, which handles both dialects ERCOT uses.
Everything else is a CSV or a workbook, read with an exact header check, so a column ERCOT adds or
renames is noticed.


In [ ]:
for emil_id in pulls:
    result = s.build_raw(emil_id)
    built = result.filter(pl.col("status") == "built")
    print(f"{emil_id:11s}", dict(result.group_by("status").len().rows()), f"{int(built['rows'].sum() or 0):,} rows")

In [ ]:
print(sorted(p.name for p in (s.data_dir / "raw").iterdir() if not p.name.startswith(".")))
s.raw("psse_bus").group_by("emil_id").len().collect()

## 4. The core layer

`build_core` turns each model into one set of keyed tables, so the CRR and DAM models can be
compared row for row. Every row carries a `snapshot_id`. The DAM posts 24 models a day and renumbers
its buses every hour, so the hour is part of the id.

- `bus`: one row per bus with its `bus_key`, an identity built from the equipment attached to the
  bus rather than from PSS/E numbers (which the DAM changes hourly) or names (which are substation
  names in the DAM). A DAM bus is one RAW record; a CRR bus is the nodes closed breakers join.
- `node`: one row per CRR node with the `bus_key` of its bus. The DAM model has none.
- `branch` and `branch_rating`: lines and transformers with a stable `branch_id`, and both rating
  sources side by side: the RAW's rates A, B and C, and the CRR's monitored-element ratings per time-of-use block.
- `contingency` and `contingency_outage`: what each contingency removes, resolved to branch and bus
  keys; a split-bus row records which end of a branch moves.
- `gtc` and `gtc_member`: the GTCs; the CRR file gives members, the DAM workbook gives daily limits.
- `settlement_point` and `settlement_point_bus`: each settlement point's buses and weights. A DAM hub
  follows Protocols 3.5.2: equal weight per hub bus, then per energized bus inside it.
- `load`: every load with its bus (and its node, for CRR), service status and MW.

Where the code decides something, the decision has a row in `docs/assumptions.md`, the **register**,
and the code names that row; a test fails when it does not.


In [ ]:
result = s.build_core()
print(dict(result.group_by("status").len().rows()))
snap = s.core("snapshot").collect()
snap.group_by("model_kind").agg(pl.len().alias("snapshots"), pl.col("snapshot_id").min().alias("first"), pl.col("snapshot_id").max().alias("last")).sort("model_kind")

In [ ]:
dam = snap.filter((pl.col("model_kind") == "dam") & (pl.col("hour") == 12)).sort("operating_date", "revision")
DAM_ID = dam["snapshot_id"][-1]  # hour 12 of the newest DAM day fetched
DAY = dam["operating_date"][-1]
monthly = snap.filter(pl.col("model_kind") == "crr_monthly").sort("month", "revision")
same_month = monthly.filter(pl.col("month") == DAY.replace(day=1))
CRR_ID = (same_month if same_month.height else monthly)["snapshot_id"][-1]
print("DAM snapshot:", DAM_ID, "| CRR snapshot:", CRR_ID)

def counts(snapshot_id):
    return {
        "buses": s.core("bus").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),
        "nodes": s.core("node").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),  # 0 for a DAM hour
        "branches": s.core("branch").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),
        "contingencies": s.core("contingency").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),
        "gtcs": s.core("gtc").filter(pl.col("snapshot_id") == snapshot_id).select(pl.len()).collect().item(),
        "settlement_points": dict(s.core("settlement_point").filter(pl.col("snapshot_id") == snapshot_id).group_by("kind").len().sort("kind").collect().rows()),
    }
pl.DataFrame([{"snapshot": "CRR", **counts(CRR_ID)}, {"snapshot": "DAM", **counts(DAM_ID)}])

## 5. Matching the two models and recording where they differ

The CRR model auctions congestion revenue rights months ahead; the DAM model clears the day-ahead
market. They describe the same grid in different vocabularies. `match_*` records identity and
`diff_*` records differences; a difference is never smoothed over.

The matchers run in this order: settlement points and generator or load names; branches by the CRR
workbook's operations name with a stated normalization; buses through the ends of matched branches,
never by name or number; contingencies by name, with their member sets compared. Every match carries
its `match_method`, and unmatched records stay in the output.


In [ ]:
mb = s.match_branches(CRR_ID, DAM_ID)
print("branches by match method:", dict(mb.group_by("match_method").len().sort("match_method").rows()))
mbus = s.match_buses(CRR_ID, DAM_ID)
print("buses by match method:", dict(mbus.group_by("match_method").len().sort("match_method").rows()))
mc = s.match_contingencies(CRR_ID, DAM_ID)
print("contingencies by match method:", dict(mc.group_by("match_method").len().sort("match_method").rows()))

In [ ]:
diff = s.diff_branches(CRR_ID, DAM_ID)
verdicts = [c for c in diff.columns if c.startswith("same_")]
print("matched branches compared:", diff.height)
diff.select([pl.col(c).mean().round(3) for c in verdicts])  # share of matched branches that agree, per aspect

In [ ]:
print("settlement point rows compared:", s.diff_settlement_points(CRR_ID, DAM_ID).height)
print("load rows compared:", s.diff_loads(CRR_ID, DAM_ID).height)

## 6. The network

`network` assembles one snapshot into the shape a DC power-flow solver reads: indexed buses, branches
with limits, contingencies as index sets, signed GTC members and settlement point weights, plus a list
of everything it dropped and why. Each reason names a register row.

The defaults do what ERCOT does. The CRR auction solves the CRR model with its closed breakers as
branches, so by default every CRR node is a bus of its own; `Options(contract_ties=True)` joins them.
Every other rule (which rating is the limit, out-of-service elements dropped) is ERCOT's practice,
fixed in code. A network is assembled from the core tables on every call.


In [ ]:
from ercot_mis.out.network import Options

net = s.network(DAM_ID)
print(net.summary())
print(Options())

## 7. Shift factors and the test against what cleared

`ercot_mis.out.shift_factors.DcSystem` is the one place shift factors are computed. It factorizes the
network once and re-solves it under a contingency, removing branches, splitting buses and handling
the part of the grid a contingency cuts off.

The repository tests its models against ERCOT's published results without re-clearing the market.
The DAM publishes the constraints that bound, their shadow prices and the prices. ERCOT's price at a
bus is the system price minus, for every binding constraint, the shadow price times the bus's shift
factor on it; the DAM has no loss term. So if the network, the names and the contingency definitions
are right, every settlement point price can be rebuilt from the published shadow prices. The
difference is the **residual**. `checks/prices.py` computes it per hour and writes
`data/reports/prices/<snapshot>.json`; the daily pull runs it on the newest days. Numbers live in
those reports, not in the documentation.


In [ ]:
from ercot_mis.out.shift_factors import DcSystem

system = DcSystem(net)
row = system.shift_factors([0])[0]  # how much of one MW at each bus flows on branch 0
print("buses:", row.size, "| largest shift factor on branch 0:", round(float(abs(row).max()), 3))

In [ ]:
from ercot_mis.checks import prices

report = prices.check_hour(s, DAM_ID)
{k: report[k] for k in ("binding_rows", "rows_by_outcome", "system_price", "residual_by_kind", "points_cut_off_by_a_binding_contingency")}

The second test, the **flow check**, pushes the day-ahead market's awards through the network and
compares the flows on the binding constraints with ERCOT's. The awards come from the 60-day disclosure,
so a day's awards arrive two months after its model. `core.hourly_award` holds every award as a signed
MW at a settlement point and `injections` sums them to the net injection per point and hour. The cell
below checks the balance: supply awards equal demand awards every hour (register rows INJ-01 and INJ-02).


In [ ]:
from datetime import datetime

from ercot_mis.core import award

awards = s.core("hourly_award").collect()
AWARD_DAY = max(datetime.strptime(d, "%m/%d/%Y").date() for d in awards["delivery_date"].unique())
day_rows = awards.filter(pl.col("delivery_date") == f"{AWARD_DAY:%m/%d/%Y}")
print("awards day:", AWARD_DAY, "| award rows:", day_rows.height, "| settlement points with a net injection:", s.injections(AWARD_DAY).height // 24)
award.hourly_balance(day_rows).select("hour_ending", "generation", "energy_only_offer", "energy_bid", "net_mw", "net_share_of_supply").head(6)

## 8. The viewer

`viewer` writes one HTML file that shows the network substation by substation, two snapshots side by
side, every branch coloured by how it compares. The file holds CEII: it stays inside the data folder
and is opened locally only. The page keeps the current substation in the URL hash; open it with
`?nohash` if the URL could leave the machine.


In [ ]:
page = s.viewer(CRR_ID, DAM_ID)
print("written:", page.relative_to(s.data_dir), f"{page.stat().st_size / 1e6:.1f} MB")

## 9. The daily pull

`scripts/daily_pull.py` runs every day through launchd (`scripts/launchd/`). It fetches every fetched
product, lists the rest, builds raw and core for whatever is new, and runs the price check on the newest
days and the flow check on the newest day with awards. It writes `data/logs/daily_pull.log` (one block
per run: a line per product, a line saying what is at risk, one line per check) and
`data/logs/last_run.json` (the run's level and the days until an EWS document is lost). A macOS
notification fires when the level is not `ok`. If the Mac slept through the scheduled time, launchd
runs the job on wake.

Where to read next: `docs/assumptions.md` (every decision, its status and the check that measures it),
`docs/datasets/*.md` (one note per dataset), `docs/out-network.md` (the network's conventions), and
the reports under `data/reports/`.


In [ ]:
import json

status = REPO / "data" / "logs" / "last_run.json"
print(json.dumps(json.loads(status.read_text()), indent=1) if status.is_file() else "no production run on this machine yet")